# The Metaprogramming Toolkit - Decorators, Closures and Caching

Difficulty: Intermediate | ~40 min | Requires basic Python (functions, classes, try/except)

A messy legacy codebase needs logging, performance tracking, security, and
caching - but you cannot touch the core functions. This lab builds four
decorators that wrap those functions and add the administrative work around
them.

In [1]:
!pip install tabulate==0.10.0


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Dell\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## Section 0 - Warm up: functions are objects

Run the cell to see how one function can wrap another, and what that wrapping costs.

In [2]:
# Functions are objects, so they can be passed in and returned. Here we
# hand greet to shout, get back a wrapper, and rebind the name greet to it.
def shout(func):
    def wrapper(name):
        return func(name).upper() + "!"
    return wrapper

def greet(name):
    return f"hello {name}"

greet = shout(greet)
print(greet("team"))
print("function name is now:", greet.__name__)

HELLO TEAM!
function name is now: wrapper


## Section 1 - The scenario

Add timing, access control, retries, and caching to messy core functions - without editing them.

In [3]:
import functools
import random
import time

## Section 2 - Part 1: the @timer decorator

Wrap any function so it prints how long it took to run.

In [4]:
# @timer measures a function's run time and prints it, then returns the
# result untouched. @functools.wraps copies the original name/docstring
# onto the wrapper so introspection keeps working.
def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} ran in {elapsed:.4f}s")
        return result
    return wrapper

@timer
def find_primes(limit):
    return [n for n in range(2, limit + 1)
            if all(n % d for d in range(2, int(n ** 0.5) + 1))]

primes = find_primes(2000)
print("primes found:", len(primes))

find_primes ran in 0.0033s
primes found: 303


## Section 3 - Part 2: the @authenticate decorator

Gate a sensitive function behind a role check using a decorator factory.

In [5]:
# @authenticate(role=...) needs an argument, so authenticate is a factory:
# calling it returns the real decorator. The wrapper reads a shared
# current_user dict and raises PermissionError if the role does not match.
current_user = {"name": "alice", "role": "admin"}

def authenticate(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user.get("role") != role:
                raise PermissionError(
                    f"{current_user['name']} needs role '{role}'")
            return func(*args, **kwargs)
        return wrapper
    return decorator

@authenticate(role="admin")
def delete_user(user_id):
    return f"Deleted user {user_id}"

print(delete_user(7))

current_user["role"] = "viewer"
try:
    delete_user(7)
except PermissionError as error:
    print("Denied:", error)

Deleted user 7
Denied: alice needs role 'admin'


## Section 4 - Part 3: the @retry decorator

Re-run a flaky function until it succeeds or the attempts run out.

In [6]:
# @retry(max_attempts=...) retries the wrapped call when it raises
# ConnectionError (a flaky network in real life). If every attempt fails,
# the last error is re-raised as a RuntimeError after the loop.
def retry(max_attempts=3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except ConnectionError as error:
                    print(f"Attempt {attempt} failed: {error}")
                    last_error = error
            raise RuntimeError(
                f"giving up after {max_attempts} attempts: {last_error}")
        return wrapper
    return decorator

flaky = {"failures_left": 2}

@retry(max_attempts=3)
def fetch_orders():
    if flaky["failures_left"] > 0:
        flaky["failures_left"] -= 1
        raise ConnectionError("gateway timed out")
    return ["order A", "order B"]

print(fetch_orders())

# Give up path: more failures than attempts.
flaky["failures_left"] = 5
try:
    fetch_orders()
except RuntimeError as error:
    print(error)

Attempt 1 failed: gateway timed out
Attempt 2 failed: gateway timed out
['order A', 'order B']
Attempt 1 failed: gateway timed out
Attempt 2 failed: gateway timed out
Attempt 3 failed: gateway timed out
giving up after 3 attempts: gateway timed out


## Section 5 - Part 4: the @cache decorator

Skip re-computation by remembering past results in a closure-held dict.

In [7]:
# @cache memoizes results. store lives in the closure, one dict per
# decorated function. key = args (a tuple, so it is hashable); on a hit
# the function body is skipped entirely.
def cache(func):
    store = {}

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        key = args
        if key not in store:
            print(f"computing {func.__name__} ...")
            store[key] = func(*args, **kwargs)
        return store[key]
    return wrapper

@cache
def expensive(n):
    time.sleep(0.02)  # pretend this is a slow computation
    return n ** 2

print(expensive(10))
print(expensive(10))  # cached: no "computing" line, returns instantly
print(expensive(12))

computing expensive ...
100
100
computing expensive ...
144


## Section 6 - The toolkit ledger

Every decorator wraps a core function without touching it - one glance at the ledger.

In [8]:
from tabulate import tabulate

ledger = [
    ["@timer", "performance tracking", "prints elapsed seconds"],
    ["@authenticate", "access control", "admin allowed, viewer denied"],
    ["@retry", "flaky-call resilience", "2 retries then success"],
    ["@cache", "avoid re-computation", "computed once, reused twice"],
]

print(tabulate(ledger, headers=["Decorator", "Concern", "Demo outcome"], tablefmt="grid"))

+---------------+-----------------------+------------------------------+
| Decorator     | Concern               | Demo outcome                 |
+===============+=======================+==============================+
| @timer        | performance tracking  | prints elapsed seconds       |
+---------------+-----------------------+------------------------------+
| @authenticate | access control        | admin allowed, viewer denied |
+---------------+-----------------------+------------------------------+
| @retry        | flaky-call resilience | 2 retries then success       |
+---------------+-----------------------+------------------------------+
| @cache        | avoid re-computation  | computed once, reused twice  |
+---------------+-----------------------+------------------------------+


## Section 7 - Optional Exercise

Extend @cache so keyword arguments are part of the cache key, then prove `expensive(10, debug=True)` computes again while repeating `expensive(10)` hits the cache.